# Tres categorías que BETO no conoce

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

| | |
|---|---|
| **Problema** | Un modelo listo clasifica sentimiento, pero no conoce educación, movilidad y salud. |
| **Producto** | Cuatro predicciones comparadas sobre los mismos avisos y una decisión de uso. |
| **Recursos** | BETO, unos 440 MB. Dos corridas breves; la carga del modelo puede tardar más que el entrenamiento. |

Una oficina recibe avisos y necesita ordenarlos en tres categorías propias. Reutilizar un modelo no significa que ya conozca esta clasificación.

**¿Qué se transfiere de BETO y qué tenemos que enseñarle con nuestros avisos?**

> **Cómo se trabaja.** Primero explorás cada resultado. Los microlaboratorios aparecen después de ver el mecanismo. Sus líneas `COMPLETAR` tienen valores neutros para que el cuaderno corra aunque no las resuelvas.

In [ ]:
import time

import pandas as pd
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EvalPrediction,
    Trainer,
    TrainingArguments,
    set_seed,
)

import utils
from utils import dibujar_mapa_de_calor

# BETO: BERT preentrenado en espanol; se descarga de Hugging Face Hub
MODELO = "dccuchile/bert-base-spanish-wwm-uncased"
SEMILLA = 42
set_seed(SEMILLA)  # fija el azar de Python, NumPy y PyTorch
utils.versiones()

## 1. Las categorías son una decisión humana

Los 36 avisos son sintéticos y traen una etiqueta humana. `train` sirve para actualizar pesos; `val`, para comparar variantes; `test`, para una única medición final. La separación estaba fijada antes de entrenar.

La etiqueta tampoco es una verdad natural: «la campaña de vacunación llegará a las escuelas» puede discutirse como salud o educación.

In [ ]:
registros = utils.cargar_json("clasificacion.json")
utils.resumen_particiones(registros)

split,test,train,val
etiqueta,,,
educación,3,6,3
movilidad,3,6,3
salud,3,6,3


> **Para mirar.** Hay seis avisos por categoría en train y tres en val y test. Con tan pocos casos, una predicción cambia mucho la métrica. Este conjunto muestra el procedimiento: no permite estimar cómo funcionaría un clasificador municipal real.

## 2. Una referencia que no viene de un Transformer

TF-IDF cuenta rasgos de las palabras de train y una regresión logística los usa para separar categorías. No reutiliza un modelo preentrenado. Es una **referencia**: si un modelo más grande cuesta más y no aporta evidencia mejor, no hay razón para elegirlo.

In [ ]:
textos_train, etiquetas_train = utils.separar(registros, "train")
textos_val, etiquetas_val = utils.separar(registros, "val")
textos_test, etiquetas_test = utils.separar(registros, "test")

inicio = time.perf_counter()
vectorizador = TfidfVectorizer()
numeros_train = vectorizador.fit_transform(textos_train)
referencia = LogisticRegression(max_iter=1000, random_state=SEMILLA)
referencia.fit(numeros_train, etiquetas_train)
tiempo_referencia = time.perf_counter() - inicio
pred_tfidf = referencia.predict(vectorizador.transform(textos_val))
f1_tfidf = f1_score(etiquetas_val, pred_tfidf, average="macro")
accuracy_tfidf = accuracy_score(etiquetas_val, pred_tfidf)
utils.tabla([{
    "accuracy_val": round(accuracy_tfidf, 3),
    "f1_macro_val": round(f1_tfidf, 3),
    "segundos": round(tiempo_referencia, 3),
}])

,accuracy_val,f1_macro_val,segundos
0,0.778,0.75,0.014


> **Para mirar.** Esta referencia ya clasifica los mismos avisos de val. El número no decide solo: después veremos dónde acierta por palabras y dónde se confunde. La comparación útil usa los mismos textos y las mismas etiquetas.

## 3. BETO base: texto, tokens y una cabeza vacía

BETO base fue preentrenado en español, no en estas tres categorías. El **tokenizador** transforma texto en IDs; el **encoder** produce una representación contextual; una **cabeza nueva** convierte esa representación en tres puntuaciones.

`texto → tokens → encoder preentrenado → cabeza nueva → educación / movilidad / salud`

In [ ]:
tokenizador = AutoTokenizer.from_pretrained(MODELO)
aviso = "El municipio instaló nuevas bicisendas en el barrio."
# input_ids: el numero que el modelo asigna a cada pieza de texto
ids = tokenizador(aviso)["input_ids"]
pd.DataFrame({
    "token": tokenizador.convert_ids_to_tokens(ids),
    "id": ids,
})

,token,id
0,[CLS],4
1,el,1039
2,municipio,3588
3,instaló,18701
4,nuevas,3390
5,bici,9119
6,##sen,5010
7,##das,1688
8,en,1035
9,el,1039


> **Para mirar.** `bicisendas` se parte en piezas. El encoder recibe IDs, no palabras. Las categorías finales todavía no existen en BETO: aparecen al crear una cabeza de tres salidas.

In [ ]:
id_de_etiqueta = {"educación": 0, "movilidad": 1, "salud": 2}
etiqueta_de_id = {0: "educación", 1: "movilidad", 2: "salud"}


def preparar_datos(textos: list[str], etiquetas: list[str]) -> Dataset:
    """Tokeniza avisos y agrega IDs de sus etiquetas.

    Args:
        textos: Avisos de una particion.
        etiquetas: Etiquetas en el mismo orden que los avisos.

    Returns:
        Dataset listo para el Trainer.
    """
    datos = dict(tokenizador(
        textos,
        truncation=True,
        padding="max_length",
        max_length=32,
    ))
    datos["label"] = []
    for etiqueta in etiquetas:
        datos["label"].append(id_de_etiqueta[etiqueta])
    return Dataset.from_dict(datos)


datos_train = preparar_datos(textos_train, etiquetas_train)
datos_val = preparar_datos(textos_val, etiquetas_val)
datos_test = preparar_datos(textos_test, etiquetas_test)

## 4. Primera estrategia: entrenar solo la cabeza

Creamos una cabeza de tres etiquetas con pesos nuevos. **Congelar** el encoder impide que cambie; la cabeza sí aprende con los 18 avisos de train y sus etiquetas. Es transferencia: conservamos lo que BETO ya aprendió de español y enseñamos una salida nueva.

| Pieza | Estado antes | Estado después |
|---|---|---|
| Encoder de BETO | preentrenado en español | queda fijo |
| Cabeza de tres etiquetas | nueva, al azar | aprende con train |

In [ ]:
def cargar_modelo() -> AutoModelForSequenceClassification:
    """Carga BETO con una cabeza nueva de tres salidas.

    Returns:
        Modelo con la misma semilla en cada corrida.
    """
    set_seed(SEMILLA)
    return AutoModelForSequenceClassification.from_pretrained(
        MODELO,
        num_labels=3,  # cabeza nueva: una salida por clase
        attn_implementation="eager",  # atencion sin optimizaciones; permite inspeccionarla
    )

In [ ]:
def calcular_metricas(prediccion: EvalPrediction) -> dict:
    """Calcula accuracy y F1 macro.

    Args:
        prediccion: Puntajes de cada etiqueta y etiquetas reales.

    Returns:
        Dos metricas sobre una particion reservada.
    """
    puntajes, reales = prediccion
    predichas = puntajes.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(reales, predichas),
        "f1_macro": f1_score(reales, predichas, average="macro"),
    }

In [ ]:
def crear_trainer(
    modelo: AutoModelForSequenceClassification,
    tasa: float,
) -> Trainer:
    """Arma un Trainer con train y val fijos.

    Args:
        modelo: Copia de BETO que se va a actualizar.
        tasa: Tamano de las actualizaciones de pesos.

    Returns:
        Trainer listo para entrenar y evaluar.
    """
    argumentos = TrainingArguments(
        output_dir="salida",
        learning_rate=tasa,
        num_train_epochs=3,
        per_device_train_batch_size=8,
        eval_strategy="epoch",
        save_strategy="no",  # no guarda checkpoints en disco
        report_to="none",  # sin servicios externos de seguimiento
        disable_tqdm=True,  # sin barras de progreso en la salida
    )
    return Trainer(
        model=modelo,
        args=argumentos,
        train_dataset=datos_train,
        eval_dataset=datos_val,
        compute_metrics=calcular_metricas,
    )

In [ ]:
modelo_cabeza = cargar_modelo()
for parametro in modelo_cabeza.base_model.parameters():
    parametro.requires_grad = False

parametros_totales = modelo_cabeza.num_parameters()
parametros_cabeza = modelo_cabeza.num_parameters(only_trainable=True)
trainer_cabeza = crear_trainer(modelo_cabeza, tasa=1e-3)

antes = trainer_cabeza.predict(datos_val)
ids_antes = antes.predictions.argmax(axis=-1)
tiempo_cabeza = trainer_cabeza.train().metrics["train_runtime"]
despues = trainer_cabeza.predict(datos_val)
ids_cabeza = despues.predictions.argmax(axis=-1)
metricas_cabeza = trainer_cabeza.evaluate()
f1_cabeza = metricas_cabeza["eval_f1_macro"]

pd.DataFrame({
    "aviso": textos_val,
    "real": etiquetas_val,
    "cabeza_nueva": pd.Series(ids_antes).map(etiqueta_de_id),
    "cabeza_entrenada": pd.Series(ids_cabeza).map(etiqueta_de_id),
})

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-uncased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'eval_loss': 1.023121953010559, 'eval_model_preparation_time': 0.0012, 'eval_accuracy': 0.5555555555555556, 'eval_f1_macro': 0.5222222222222221, 'eval_runtime': 0.0489, 'eval_samples_per_second': 183.988, 'eval_steps_per_second': 40.886, 'epoch': 1.0}
{'eval_loss': 0.9829432368278503, 'eval_model_preparation_time': 0.0012, 'eval_accuracy': 0.5555555555555556, 'eval_f1_macro': 0.5222222222222221, 'eval_runtime': 0.0389, 'eval_samples_per_second': 231.22, 'eval_steps_per_second': 51.382, 'epoch': 2.0}
{'eval_loss': 0.9694808125495911, 'eval_model_preparation_time': 0.0012, 'eval_accuracy': 0.5555555555555556, 'eval_f1_macro': 0.5222222222222221, 'eval_runtime': 0.0392, 'eval_samples_per_second': 229.457, 'eval_steps_per_second': 50.991, 'epoch': 3.0}
{'train_runtime': 0.4134, 'train_samples_per_second': 130.633, 'train_steps_per_second': 21.772, 'train_loss': 1.006966061062283, 'epoch': 3.0}
{'eval_loss': 0.9694808125495911, 'eval_model_preparation_time': 0.0012, 'eval_accuracy': 0.5555

,aviso,real,cabeza_nueva,cabeza_entrenada
0,El curso de programación tendrá encuentros semanales.,educación,educación,educación
1,El centro cultural abre un taller de fotografía.,educación,movilidad,movilidad
2,La biblioteca ofrece encuentros de lectura para adolescentes.,educación,movilidad,salud
3,El tren agrega frecuencias durante el fin de semana.,movilidad,movilidad,movilidad
4,Una avenida permanece cerrada por mantenimiento.,movilidad,movilidad,movilidad
5,El barrio solicita mejorar la iluminación de las paradas.,movilidad,movilidad,movilidad
6,La campaña de vacunación llegará a las escuelas.,salud,educación,movilidad
7,El consultorio abre nuevos turnos para controles.,salud,movilidad,movilidad
8,El equipo de salud dará una charla sobre prevención.,salud,movilidad,salud


> **Para mirar.** La cabeza nueva no conocía las categorías; las predicciones posteriores usan los mismos avisos de val. Mirá el texto y la etiqueta real: que cambie una salida no prueba que mejoró. Solo son entrenables unos pocos miles de parámetros frente a los más de cien millones de BETO.

### Microlaboratorio 1 — Congelar y contar

**Tarea.** Congelá una copia de BETO y contá qué parámetros siguen aprendiendo.  
**Producto.** `entrenables`, comprobado.  
**Antes de empezar.** La primera estrategia.

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) primer bucle | el encoder congelado | `parametro.requires_grad = False` |
| (2) segundo bucle | suma de parámetros entrenables | `if parametro.requires_grad` y `parametro.numel()` |

**Comprobación.** La cabeza debe quedar entrenable, pero representar menos del 1 % del total.

> **Ayuda 1.** Repetí el primer bucle de la sección 4.  
> **Ayuda 2.** `numel()` cuenta los números de un parámetro.

In [ ]:
modelo_de_prueba = cargar_modelo()
for parametro in modelo_de_prueba.base_model.parameters():
    pass  # COMPLETAR (1): congelar el encoder

total = modelo_de_prueba.num_parameters()
entrenables = 0
for parametro in modelo_de_prueba.parameters():
    pass  # COMPLETAR (2): sumar los que se entrenan

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-uncased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
utils.comprobar(
    entrenables > 0 and entrenables < total / 100,
    "revisa: congela el encoder y suma los parametros entrenables",
)

Todavía no: revisa: congela el encoder y suma los parametros entrenables


## 5. Segunda estrategia: ajustar también el encoder

Cargamos **otra copia del mismo BETO base** y la misma cabeza inicial. Esta vez no congelamos el encoder: también cambia con los mismos 18 avisos. Bajamos la tasa de `1e-3` a `2e-5`, porque ahora actualizamos pesos preentrenados.

Esta comparación cambia dos cosas —pesos entrenables y tasa—. Muestra estrategias viables, pero no aísla una causa única.

In [ ]:
modelo_completo = cargar_modelo()
parametros_completo = modelo_completo.num_parameters(only_trainable=True)
trainer_completo = crear_trainer(modelo_completo, tasa=2e-5)
tiempo_completo = trainer_completo.train().metrics["train_runtime"]
salida_completo = trainer_completo.predict(datos_val)
ids_completo = salida_completo.predictions.argmax(axis=-1)
metricas_completo = trainer_completo.evaluate()
f1_completo = metricas_completo["eval_f1_macro"]
utils.tabla([{
    "parametros_entrenables": parametros_completo,
    "f1_macro_val": round(f1_completo, 3),
    "segundos": round(tiempo_completo, 2),
}])

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-uncased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'eval_loss': 0.9618323445320129, 'eval_accuracy': 0.5555555555555556, 'eval_f1_macro': 0.5222222222222221, 'eval_runtime': 0.3553, 'eval_samples_per_second': 25.329, 'eval_steps_per_second': 5.629, 'epoch': 1.0}


{'eval_loss': 0.8639652729034424, 'eval_accuracy': 0.7777777777777778, 'eval_f1_macro': 0.7380952380952381, 'eval_runtime': 0.3637, 'eval_samples_per_second': 24.743, 'eval_steps_per_second': 5.498, 'epoch': 2.0}


{'eval_loss': 0.8299213647842407, 'eval_accuracy': 0.7777777777777778, 'eval_f1_macro': 0.75, 'eval_runtime': 0.3638, 'eval_samples_per_second': 24.736, 'eval_steps_per_second': 5.497, 'epoch': 3.0}
{'train_runtime': 3.8418, 'train_samples_per_second': 14.056, 'train_steps_per_second': 2.343, 'train_loss': 0.8634890980190701, 'epoch': 3.0}
{'eval_loss': 0.8299213647842407, 'eval_accuracy': 0.7777777777777778, 'eval_f1_macro': 0.75, 'eval_runtime': 0.0472, 'eval_samples_per_second': 190.573, 'eval_steps_per_second': 42.35, 'epoch': 3.0}


,parametros_entrenables,f1_macro_val,segundos
0,109853187,0.75,3.84


> **Para mirar.** Ahora aprenden alrededor de 110 millones de parámetros. Más capacidad no garantiza mejores categorías: leé la tabla de casos antes de elegir. Con nueve avisos de val, un error cambia mucho la métrica.

## 6. Los mismos avisos, cuatro miradas

La tabla pone lado a lado TF-IDF, la cabeza nueva, la cabeza entrenada y BETO completo. Las categorías «reales» son decisiones humanas discutibles; si dos métodos discrepan, leé el aviso antes de llamar error a uno de ellos.

In [ ]:
tabla_comparativa = pd.DataFrame({
    "aviso": textos_val,
    "real": etiquetas_val,
    "TF_IDF": pred_tfidf,
    "cabeza_nueva": pd.Series(ids_antes).map(etiqueta_de_id),
    "cabeza_entrenada": pd.Series(ids_cabeza).map(etiqueta_de_id),
    "BETO_completo": pd.Series(ids_completo).map(etiqueta_de_id),
})
tabla_comparativa

,aviso,real,TF_IDF,cabeza_nueva,cabeza_entrenada,BETO_completo
0,El curso de programación tendrá encuentros semanales.,educación,salud,educación,educación,educación
1,El centro cultural abre un taller de fotografía.,educación,salud,movilidad,movilidad,educación
2,La biblioteca ofrece encuentros de lectura para adolescentes.,educación,educación,movilidad,salud,educación
3,El tren agrega frecuencias durante el fin de semana.,movilidad,movilidad,movilidad,movilidad,movilidad
4,Una avenida permanece cerrada por mantenimiento.,movilidad,movilidad,movilidad,movilidad,movilidad
5,El barrio solicita mejorar la iluminación de las paradas.,movilidad,movilidad,movilidad,movilidad,movilidad
6,La campaña de vacunación llegará a las escuelas.,salud,salud,educación,movilidad,educación
7,El consultorio abre nuevos turnos para controles.,salud,salud,movilidad,movilidad,educación
8,El equipo de salud dará una charla sobre prevención.,salud,salud,movilidad,salud,salud


In [ ]:
nombres_modelos = [
    "TF_IDF",
    "cabeza_nueva",
    "cabeza_entrenada",
    "BETO_completo",
]
coincidencias_por_aviso = []
nombres_de_avisos = []
numero_aviso = 0
for registro in tabla_comparativa.to_dict(orient="records"):
    numero_aviso += 1
    coincidencias = []
    for nombre_modelo in nombres_modelos:
        coincide = registro[nombre_modelo] == registro["real"]
        coincidencias.append(int(coincide))
    coincidencias_por_aviso.append(coincidencias)
    nombres_de_avisos.append(f"Aviso {numero_aviso}")

dibujar_mapa_de_calor(
    matriz=coincidencias_por_aviso,
    filas=nombres_de_avisos,
    columnas=nombres_modelos,
    titulo="Coincidencia con la etiqueta de referencia",
    formato="{:.0f}",
    etiqueta_filas="Avisos de validacion",
    etiqueta_columnas="Estrategia",
)

> **Para mirar.** `1` indica que la predicción coincide con la etiqueta de referencia; `0`, que difiere. La matriz muestra dónde discrepan las estrategias, pero no convierte una etiqueta discutible en verdad ni mide rendimiento futuro. Volvé a la tabla para leer cada caso.

In [ ]:
if f1_completo > f1_cabeza:
    trainer_elegido = trainer_completo
    nombre_elegido = "BETO completo"
else:
    trainer_elegido = trainer_cabeza
    nombre_elegido = "cabeza entrenada"

print("Variante neuronal elegida por val:", nombre_elegido)
resultados_test = trainer_elegido.evaluate(datos_test)
utils.tabla([resultados_test])[["eval_accuracy", "eval_f1_macro"]]

Variante neuronal elegida por val: BETO completo


{'eval_loss': 0.8937973380088806, 'eval_accuracy': 0.6666666666666666, 'eval_f1_macro': 0.6793650793650793, 'eval_runtime': 0.0567, 'eval_samples_per_second': 158.73, 'eval_steps_per_second': 35.273, 'epoch': 3.0}


,eval_accuracy,eval_f1_macro
0,0.666667,0.679365


> **Para mirar.** Test se abre una sola vez, después de elegir por val entre las dos variantes neuronales. Esta regla no obliga a elegir BETO frente a TF-IDF: la decisión final compara las tres opciones. Los 36 avisos no alcanzan para afirmar rendimiento futuro.

## 7. Decidir con evidencia

La tabla final compara costo y F1 de val. Un decimal no alcanza: mirá dónde discrepan y qué harías con esos avisos.

In [ ]:
pd.DataFrame({
    "estrategia": ["TF-IDF", "solo cabeza", "BETO completo"],
    "f1_macro_val": [f1_tfidf, f1_cabeza, f1_completo],
    "segundos": [tiempo_referencia, tiempo_cabeza, tiempo_completo],
    "parametros_entrenables": [
        referencia.coef_.size + referencia.intercept_.size,
        parametros_cabeza,
        parametros_completo,
    ],
}).round(3)

,estrategia,f1_macro_val,segundos,parametros_entrenables
0,TF-IDF,0.750,0.014,294
1,solo cabeza,0.522,0.413,2307
2,BETO completo,0.750,3.842,109853187


> **Para mirar.** En este recorte, una referencia simple puede competir con BETO. No sería razonable elegir el modelo más caro solo por ser un Transformer. Con más datos y otra tarea, la conclusión podría cambiar.

### Microlaboratorio 2 — Un desacuerdo con consecuencias

**Tarea.** Elegí un aviso donde discrepen TF-IDF, la cabeza entrenada o BETO completo. ¿Qué categoría sostendrías? ¿Se justifica el modelo grande?  
**Producto.** Una fila y una decisión con dos predicciones.  
**Antes de empezar.** Las tablas de las secciones 6 y 7.

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) `mi_indice` | fila con predicciones diferentes | índice visible en `tabla_comparativa` |
| (2) `mi_decision` | categoría, dos salidas y condición | una oración de al menos veinte palabras |

**Comprobación.** El código verifica el desacuerdo; contrastá la justificación con el texto.

> **Ayuda 1.** Mirá «vacunación en las escuelas» o «taller de fotografía».  
> **Ayuda 2.** Escribí el índice de esa fila.

In [ ]:
mi_indice = -1  # COMPLETAR (1): fila con desacuerdo
mi_decision = ""  # COMPLETAR (2): evidencia y condicion

In [ ]:
utils.comprobar(
    0 <= mi_indice < len(tabla_comparativa)
    and len(set(tabla_comparativa.iloc[mi_indice][[
        "TF_IDF", "cabeza_entrenada", "BETO_completo"
    ]])) > 1,
    "revisa: elige una fila donde discrepen dos estrategias",
)
utils.comprobar(
    len(mi_decision.split()) >= 20,
    "revisa: cita el aviso, dos salidas y una condicion",
)

Todavía no: revisa: elige una fila donde discrepen dos estrategias
Todavía no: revisa: cita el aviso, dos salidas y una condicion


## Qué te llevás

- BETO aporta un encoder de español; la cabeza nueva aprende las categorías propias con avisos etiquetados.
- Congelar conserva el encoder; abrirlo permite modificarlo, pero no garantiza una mejor decisión.
- Con pocos datos, los textos donde discrepan los modelos, el costo y las etiquetas discutibles importan tanto como una métrica.

**Sigue:** en el cuaderno 03 se adapta el lenguaje de BETO a un dominio sin etiquetas, usando estrofas de *Martín Fierro*.